# Stage 5: Generate Final Resumen/Texto Pairs

**Purpose**: Produces the final short-format (resumen) and long-format (texto) descriptions by substituting all remaining placeholders.

**Input**: `data/intermediate/OBRA CIVIL/OBRA CIVIL_stage4.json`  
**Output**: `data/intermediate/OBRA CIVIL/OBRA CIVIL_stage5.json`

## What this notebook does

After text variables are resolved (Stage 4), the `resumen` and `texto` template strings still contain placeholder references like `$A` or `$L(a,%C)`. This stage:

1. Replaces function calls (e.g., `$L(a,%C)`) with their evaluated values
2. Replaces simple variable references (e.g., `$A`) with parameter values
3. Cleans up formatting artifacts (backslashes, extra whitespace)

### Output structure

Each item now contains fully instantiated text:

```json
{
  "OEB020$AAAAAA": {
    "parent_key": "OEB020$",
    "ud": "m",
    "concept": "Concrete-encased PVC conduit, 110 mm",
    "resumen": "Canalización hormigonada de 2 tubos PVC 110 mm, terreno normal...",
    "texto": "Canalización hormigonada de 2 tubos de polietileno libre de halógenos...",
    "parameters": { ... }
  }
}
```

In [7]:
!pip install ijson

In [12]:
import os
import json
import ijson
import re
import time
from typing import Iterator, Dict, Any, TextIO
import traceback

def get_variable_value(var_name, function_params, text_vars, params):
    """
    Get value for a variable or function from text_variables or parameters
    """
    # Handle function calls like L(a,%C)
    if function_params:
        if var_name in text_vars:
            # Get indices from function parameters
            indices = []
            for param in function_params.split(','):
                param = param.strip()
                if param.startswith('%'):               
                    param_key = param[1:]
                    if param_key in params and params[param_key]['values']:
                        label = params[param_key]['values'][0]['label']
                        if label.isalpha():
                            indices.append(ord(label.lower()) - ord('a'))
                        else:
                            indices.append(0)
                    else:
                        indices.append(0)
                elif param.isalpha():
                    indices.append(ord(param.lower()) - ord('a'))
                else:
                    try:
                        indices.append(int(param))
                    except ValueError:
                        indices.append(0)
            
            # Navigate through the evaluated data using indices
            result = text_vars[var_name]['evaluated']
            for idx in indices:
                if isinstance(result, list) and idx < len(result):
                    result = result[idx]
            return str(result).strip('"')
        
        return ''

    # Handle simple variables like $A
    if var_name in params:
        return str(params[var_name]['values'][0]['value'])
    elif var_name in text_vars:
        result = text_vars[var_name]['evaluated']
        if isinstance(result, list):
            result = result[0]
        return str(result).strip('"')
    
    return ''

def evaluate_string(text, text_vars, params):
    """
    Evaluate a string by replacing all placeholders with their values
    """
    result = text
    
    # Replace function calls (like L(a,%C))
    pattern = r'\$([A-Z])\((.*?)\)'
    matches = list(re.finditer(pattern, result))
    for match in reversed(matches):  # Process from end to start to avoid nested issues
        var_name = match.group(1)
        function_params = match.group(2)
        replacement = get_variable_value(var_name, function_params, text_vars, params)
        result = result[:match.start()] + replacement + result[match.end():]
    
    # Replace simple variables (like $A)
    pattern = r'\$([A-Z])'
    matches = list(re.finditer(pattern, result))
    for match in reversed(matches):
        var_name = match.group(1)
        replacement = get_variable_value(var_name, '', text_vars, params)
        result = result[:match.start()] + replacement + result[match.end():]
    
    # Remove any remaining backslashes
    result = result.replace('\\', '')
    
    return result.strip()

def process_json(input_data):
    """
    Process the input JSON and evaluate resumen and texto fields
    """   
    for key, item in input_data.items():
        if isinstance(item, dict):
            if 'resumen' in item and 'texto' in item:
                text_vars = item.get('text_variables', {})
                params = item.get('parameters', {})
                
                # Evaluate resumen and texto
                item['resumen'] = evaluate_string(item['resumen'], text_vars, params)
                item['texto'] = evaluate_string(item['texto'], text_vars, params)
    
    return input_data

def filter_fields(data):
    """
    Keep only specified fields in the JSON
    """
    filtered_data = {}
    fields_to_keep = ['parent_key', 'ud', 'concept', 'resumen', 'texto', 'parameters']
    
    for key, item in data.items():
        if isinstance(item, dict):
            filtered_item = {field: item[field] for field in fields_to_keep if field in item}
            filtered_data[key] = filtered_item
    
    return filtered_data

# Generate a structured filename
# Input: Source file path, stage number, output directory, and file extension.
# Output: A structured file name for the output file.
def generate_filename(input_file, stage, output_dir, extension="json"):
    """Generate a structured filename with stage and timestamp."""
    base_name = os.path.splitext(os.path.basename(input_file))[0]  # Get base name of the source file
    file_name = f"{base_name}_stage{stage}.{extension}"
    return os.path.join(output_dir, file_name)

# Create output directories dynamically within the source file's path
# Input: Source file path.
# Output: Path to the created output directory.

def create_output_dirs(input_file):
    """Create and return the output directory path within the source file's directory."""
    input_dir = os.path.dirname(input_file)
    output_dir = input_dir # Same directory
    os.makedirs(output_dir, exist_ok=True)  # Ensure the directory exists
    return output_dir

CHUNK_SIZE = 1000  # Adjust based on available memory

def count_total_items(filename: str) -> int:
    """Count the total number of items in the top-level JSON object."""
    with open(filename, 'r', encoding='utf-8') as f:
        return sum(1 for _ in ijson.kvitems(f, ''))

def read_json_in_chunks(filename: str, chunk_size: int) -> Iterator[Dict[str, Any]]:
    """Read large JSON file in chunks using ijson."""
    total_items = count_total_items(filename)
    items_processed = 0
    
    with open(filename, 'r', encoding='utf-8') as f:
        chunk = {}
           
        for key, value in ijson.kvitems(f,''):
            chunk[key] = value
            items_processed += 1
            
            if len(chunk) >= chunk_size:
                print(f"Yielding chunk of {len(chunk)} items. Progress: {items_processed}/{total_items}")
                yield chunk
                chunk = {}
        
        if chunk:
            print(f"Yielding final chunk of {len(chunk)} items. Total processed: {items_processed}")
            yield chunk

def process_chunk(chunk: Dict[str, Any], output_dir: str, chunk_num: int) -> str:
    """Process a single chunk of data."""
    print(f"Processing chunk {chunk_num} with {len(chunk)} items")
    transformed_data = process_json(chunk)
    final_data = filter_fields(transformed_data)
    
    chunk_file = os.path.join(output_dir, f'chunk_{chunk_num}.json')
    with open(chunk_file, 'w', encoding='utf-8') as f:
        json.dump(final_data, f, ensure_ascii=False, indent=4)
    
    return chunk_file

def merge_chunks(chunk_files: list, output_file: str):
    """Merge processed chunks into final output."""
    print(f"Merging {len(chunk_files)} chunks")
    
    with open(output_file, 'w', encoding='utf-8') as out_file:   
        out_file.write("{")  # Start JSON object    
    
        for i, chunk_file in enumerate (chunk_files):
            with open(chunk_file, 'r', encoding='utf-8') as f:
                chunk_data = f.read().strip()
                if i > 0: # add comma between chunks
                    out_file.write(", ")
                out_file.write(chunk_data[1:-1]) # Strip enclosing braces from chunk
            os.remove(chunk_file) # delete chunk file

        out_file.write("}") # End JSON object
    
    print(f"Merging completed")

def main(input_file):
    """Process large JSON file in chunks."""
    start_time = time.time()
    stage4_input = generate_filename(input_file, stage=4, output_dir=os.path.dirname(input_file))    
    output_dir = create_output_dirs(input_file)  # Create the output directory, if necessary
    chunk_files = []
    
    try:
        print(f"Reading from {stage4_input}")
        for i, chunk in enumerate(read_json_in_chunks(stage4_input, CHUNK_SIZE),1):
            chunk_file = process_chunk(chunk, output_dir, i)
            chunk_files.append(chunk_file)

        #Generate output file name
        stage5_output = generate_filename(input_file, stage=5, output_dir=output_dir)  # Generate output file name
        merge_chunks(chunk_files, stage5_output)

        print(f"Processing completed in {time.time() - start_time:.2f} seconds")
        print(f"Output saved to {stage5_output}")

    except Exception as e:
        print(f"Error during processing:")
        traceback.print_exc()

In [13]:
from utils import config

input_file = config.chapter_path("OBRA CIVIL")
main(input_file)

Reading from /work/data/intermediate/OBRA CIVIL/OBRA CIVIL_stage4.json
Yielding chunk of 1000 items. Progress: 1000/126938
Processing chunk 1 with 1000 items
Yielding chunk of 1000 items. Progress: 2000/126938
Processing chunk 2 with 1000 items
Yielding chunk of 1000 items. Progress: 3000/126938
Processing chunk 3 with 1000 items
Yielding chunk of 1000 items. Progress: 4000/126938
Processing chunk 4 with 1000 items
Yielding chunk of 1000 items. Progress: 5000/126938
Processing chunk 5 with 1000 items
Yielding chunk of 1000 items. Progress: 6000/126938
Processing chunk 6 with 1000 items
Yielding chunk of 1000 items. Progress: 7000/126938
Processing chunk 7 with 1000 items
Yielding chunk of 1000 items. Progress: 8000/126938
Processing chunk 8 with 1000 items
Yielding chunk of 1000 items. Progress: 9000/126938
Processing chunk 9 with 1000 items
Yielding chunk of 1000 items. Progress: 10000/126938
Processing chunk 10 with 1000 items
Yielding chunk of 1000 items. Progress: 11000/126938
Proc